[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [PyTorch and TensorFlow, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/pytorch-and-tensorflow-deep-dive.html)

# A Complete Project


## What you will be able to do

Take a question from raw data to a saved classifier that answers for a new station, in PyTorch and
again in Keras, with the test stations used once and everything the classifier needs saved with it.

## The idea

### The problem

The question is whether a weather station will have snow on the ground on 1 May, from where it
stands: its latitude, its longitude, its height and its distance from the coast. The other
notebooks of this guide each took one step of the answer and made the rest easy. A whole project
takes every step, in an order where each one can go wrong without an error: a split made after
looking at the data, a scale fitted on every station, a stopping point chosen on the test stations,
or a saved model that cannot be used without numbers that were never saved.

Here are the steps in an order that avoids each of those mistakes. Split the stations once, before
anything is fitted. Measure the simplest guess. Fit the scaling and the model on the training
stations, choose when to stop on the validation stations, and measure the test stations once at the
end. Then save the model with everything it needs, and load it to answer.

### A formal definition

> A model's **test set** is data set aside before anything is fitted, and used once, at the end, to
> estimate how the finished model will do on data it has never seen. Every choice made while
> building the model, including when to stop training, is made on a separate **validation set**,
> and every number the model learns from data, including the means it standardizes with, comes from
> the **training set**.

### Why it works that way

A test score is an honest estimate only of a model that never saw the test data and was never chosen
for it. Each look that changes a choice makes the next score a little more flattering, so the
looks go to the validation stations, which exist to be looked at, and the test stations are kept
for the end.

The means and standard deviations that standardize the features are learned from data, like the
weights, so they are part of the model. They come from the training stations, and a saved model that
leaves them out cannot be used. The PyTorch classifier here keeps them outside the network, so they
are saved beside its state dict by hand. A module can also hold them as buffers, registered with
`register_buffer`, and its state dict then carries them. Keras can keep them inside, in a
`Normalization` layer, so a `.keras` file carries them.

### Where this shows up

Each step comes from another notebook of this guide: batches from the **Batches and Data Pipelines**
notebook, logits and `BCEWithLogitsLoss` from the **Losses and Optimizers** notebook, early stopping
from the **Overfitting and Regularization** notebook, and the checkpoint from the **Saving and
Loading Models** notebook. The **scikit-learn, Deep Dive** guide asks the same question of the same
made-up stations with a logistic regression.

### What this notebook covers

- one split, made first, and the baseline of always guessing no snow
- scaling learned from the training stations
- a PyTorch classifier with a validation set and early stopping, and a linear one beside it
- the test stations, once
- saving the classifier with its scaling, and answering for a planned station
- the same project in Keras, with the scaling inside the model
- three mistakes that raise no error, or only a warning

### A first look

Before any of the detail, here is the whole idea in a few lines. There is nothing to run yet: read it,
and read the output underneath it. Everything from Setup onward is where you start running things.

```python
import numpy as np
import keras

train = np.array([[10.0], [20.0], [30.0]], dtype="float32")
scale = keras.layers.Normalization()
scale.adapt(train)
print(scale(np.array([[20.0], [40.0]], dtype="float32")).numpy().ravel())
```

```
[0.        2.4494898]
```

`adapt` learned the training values' mean, 20, and their standard deviation, about 8.16, and the
layer scales every value it is given later with those two numbers, so 20 becomes 0 and 40 becomes
(40 − 20) ÷ 8.16.

## Setup

Three libraries, a way to quiet TensorFlow while it loads, the stations, and a folder for the saved
classifiers.

- `numpy` builds the stations, splits them and standardizes them
- `torch` is PyTorch
- `tensorflow` is TensorFlow, and `keras` its library of layers and training tools
- `contextlib` and `os` hide the messages TensorFlow prints while it loads, as in the **Why PyTorch
  and TensorFlow** notebook
- `copy` keeps the best weights found during training
- `Path` names the saved files, and `shutil` removes their folder at the end
- `warnings` catches the warning in the last of the Common errors

`station_network_arrays` makes the 1,200 made-up weather stations of the **Why PyTorch and
TensorFlow** notebook, from a generator started from a fixed seed. `features` holds each station's
latitude, longitude, elevation in meters and distance from the coast in kilometers, in the order
`FEATURES` names them, and `snow_on_may_1` holds 1 for a station with snow on 1 May and 0 otherwise.
Nothing is split or scaled yet, since that is the project's first step. The saved classifiers go in
`scratch`, which the last cell removes.

In [1]:
import contextlib
import copy
import os
import shutil
import warnings
from pathlib import Path

import numpy as np
import torch


@contextlib.contextmanager
def c_messages_hidden():
    """Send what C and C++ code writes to the error stream to os.devnull while the block runs."""
    saved = os.dup(2)
    devnull = os.open(os.devnull, os.O_WRONLY)
    os.dup2(devnull, 2)
    try:
        yield
    finally:
        os.dup2(saved, 2)
        os.close(devnull)
        os.close(saved)


os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
with c_messages_hidden():
    import tensorflow as tf
    import keras


def station_network_arrays(n=1200, seed=2025):
    """The made-up stations of the scikit-learn guide, as arrays: four features and two targets."""
    rng = np.random.default_rng(seed)
    latitude = rng.uniform(58.0, 71.0, n)
    longitude = 5 + 1.5 * (latitude - 58) + rng.uniform(0, 5, n)
    coast_km = np.minimum(rng.exponential(25, n), 250)
    elevation = np.clip(rng.gamma(1.5, 150, n) + 3 * coast_km, 0, 1800)
    shift = np.where(coast_km < 3, 0.8,
            np.where(elevation > 700, -0.5,
            np.where((coast_km < 30) & (rng.uniform(size=n) < 0.6), 0.3, 0.0)))
    rng.choice(4, n, p=[0.4, 0.3, 0.1, 0.2])   # each station's owner, which this guide leaves out
    rng.integers(1950, 2025, n)                 # and the year it was installed
    jan_mean = (1.5 - 0.55 * (latitude - 58) - 0.006 * elevation - 4.0 * (1 - np.exp(-coast_km / 40))
                + shift + rng.normal(0, 0.9, n))
    may_mean = jan_mean + 8.5 + 0.004 * coast_km + rng.normal(0, 1.2, n)
    features = np.column_stack([latitude.round(2), longitude.round(2), elevation.round(0), coast_km.round(1)])
    return features.astype("float32"), jan_mean.round(1).astype("float32"), (may_mean < 2.5).astype("float32")


FEATURES = ["latitude", "longitude", "elevation_m", "coast_km"]
features, jan_mean, snow_on_may_1 = station_network_arrays()
SCRATCH = Path("scratch")
SCRATCH.mkdir(exist_ok=True)

print("torch", torch.__version__.split("+")[0], "| tensorflow", tf.__version__, "| keras", keras.__version__,
      "|", len(features), "stations")

torch 2.11.0 | tensorflow 2.21.0 | keras 3.13.2 | 1200 stations


## Worked examples

### The question

In [2]:
print(features.shape, features.dtype)
print("share of stations with snow on 1 May:", round(float(snow_on_may_1.mean()), 4))

(1200, 4) float32
share of stations with snow on 1 May: 0.3875


1,200 stations, and 38.75% of them have snow on 1 May.

### Split the stations before anything else

The stations are split at random, 800 for training, 200 for validation and 200 for testing, before
anything is computed from them. The seed is the one the other notebooks use, so the rows are the same
ones their `train_rows`, `valid_rows` and `test_rows` hold.

In [3]:
order = np.random.default_rng(0).permutation(len(features))
train_rows, valid_rows, test_rows = order[:800], order[800:1000], order[1000:]
for name, rows in [("training", train_rows), ("validation", valid_rows), ("test", test_rows)]:
    print(f"{name:10}", len(rows), "stations | share with snow", round(float(snow_on_may_1[rows].mean()), 3))

training   800 stations | share with snow 0.374
validation 200 stations | share with snow 0.365
test       200 stations | share with snow 0.465


The test stations happen to have more snow, 46.5%, than the training stations, 37.4%. A random draw
of 200 stations can come out that way, and the test score is measured on these 200 as they are.
Nothing in the project is chosen from these shares.

### The baseline to beat

The simplest classifier guesses the answer most training stations have, no snow, for every station.
The finished model's test accuracy has to be read beside this one's on the same test stations, so it
is measured here, and nothing is chosen from it.

In [4]:
baseline_accuracy = float((snow_on_may_1[test_rows] == 0).mean())
print("always no snow, test accuracy:", round(baseline_accuracy, 3))
print("always no snow, training accuracy:", round(float((snow_on_may_1[train_rows] == 0).mean()), 3))

always no snow, test accuracy: 0.535
always no snow, training accuracy: 0.626


Guessing no snow is right for 53.5% of the test stations, less than the 62.6% it would score on the
training stations, because the test stations have more snow.

### Scaling learned from the training stations

In [5]:
mean, std = features[train_rows].mean(axis=0), features[train_rows].std(axis=0)


def as_tensors(rows):
    return (torch.from_numpy((features[rows] - mean) / std),
            torch.from_numpy(snow_on_may_1[rows]).unsqueeze(1))


X_train, y_train = as_tensors(train_rows)
X_valid, y_valid = as_tensors(valid_rows)
X_test, y_test = as_tensors(test_rows)
print(mean.round(2), std.round(2))

[ 64.45  17.23 295.25  23.94] [  3.7    5.7  196.22  24.08]


`mean` and `std` come from the 800 training stations, and the validation and test stations are
scaled with the same two arrays, as any new station will be.

### A PyTorch classifier with early stopping

`make_classifier` builds a network with one hidden layer, or with `hidden=0` a single `Linear`
layer, which is a logistic regression. Both give one logit per station. `fit` trains in shuffled
batches of 32 with `BCEWithLogitsLoss`, measures the loss on the validation stations after every
epoch, keeps a copy of the weights from the best epoch, and stops when 15 epochs have passed without
a better one. It prints the validation loss every 10 epochs.

In [6]:
def make_classifier(hidden=16):
    if hidden == 0:
        return torch.nn.Linear(4, 1)
    return torch.nn.Sequential(torch.nn.Linear(4, hidden), torch.nn.ReLU(), torch.nn.Linear(hidden, 1))


def fit(model, max_epochs=200, patience=15):
    loader = torch.utils.data.DataLoader(torch.utils.data.TensorDataset(X_train, y_train), batch_size=32,
                                         shuffle=True, generator=torch.Generator().manual_seed(0))
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
    loss_fn = torch.nn.BCEWithLogitsLoss()
    best_loss, best_epoch, best_state = float("inf"), 0, None
    for epoch in range(1, max_epochs + 1):
        model.train()
        for features_batch, labels_batch in loader:
            optimizer.zero_grad()
            loss_fn(model(features_batch), labels_batch).backward()
            optimizer.step()
        model.eval()
        with torch.no_grad():
            valid_loss = loss_fn(model(X_valid), y_valid).item()
        if valid_loss < best_loss:
            best_loss, best_epoch, best_state = valid_loss, epoch, copy.deepcopy(model.state_dict())
        if epoch % 10 == 0:
            print(epoch, "validation loss", round(valid_loss, 3))
        if epoch - best_epoch >= patience:
            break
    model.load_state_dict(best_state)
    print("stopped after epoch", epoch, "| best epoch", best_epoch, "| validation loss", round(best_loss, 3))


def accuracy(model, X, y):
    with torch.no_grad():
        return ((model(X) > 0).float() == y).float().mean().item()


torch.manual_seed(0)
net = make_classifier()
fit(net)

10 validation loss 0.256
20 validation loss 0.254
stopped after epoch 29 | best epoch 14 | validation loss 0.252


The network's best validation loss came at epoch 14. `fit` went on for 15 more epochs without
beating it, stopped after epoch 29 and put back the weights from epoch 14.

In [7]:
torch.manual_seed(0)
linear = make_classifier(hidden=0)
fit(linear)

10 validation loss 0.301
20 validation loss 0.278
30 validation loss 0.272
40 validation loss 0.272
50 validation loss 0.271
60 validation loss 0.269
70 validation loss 0.27
80 validation loss 0.27
90 validation loss 0.269
stopped after epoch 92 | best epoch 77 | validation loss 0.267


The linear classifier learns more slowly, and its validation loss went on improving until epoch 77.
Its best, 0.267, is above the network's 0.252, so the network is the classifier this project keeps,
chosen on the validation stations before the test stations are measured.

### The test stations measured once

`accuracy` counts a station as a snow station when its logit is above 0, which is a probability above
one half. Every choice is made now, so the test stations are measured, and the validation accuracy
is printed beside them.

In [8]:
print(f"{'always no snow':15} test {baseline_accuracy:.3f}")
for name, model in [("linear", linear), ("network", net)]:
    print(f"{name:15} test {accuracy(model, X_test, y_test):.3f} | validation {accuracy(model, X_valid, y_valid):.3f}")

always no snow  test 0.535
linear          test 0.805 | validation 0.870
network         test 0.820 | validation 0.875


Both beat the baseline's 0.535 by a wide margin. The network gets 0.82 of the test stations right
and the linear classifier 0.805, a difference of 3 stations in 200, which is too few to say that one
is better. Both score lower on the test stations than on the validation stations, which is what a
different draw of 200 stations, with more snow in it, can do.

### Saving the classifier with its scaling

The checkpoint holds the network's state dict, the number of hidden units it was built with, and the
training stations' means and standard deviations as tensors. `load_classifier` builds the network
again from those, and returns a function that takes a station's features as they come and gives the
probability of snow.

In [9]:
torch.save({"model": net.state_dict(), "hidden": 16,
            "mean": torch.from_numpy(mean), "std": torch.from_numpy(std)},
           SCRATCH / "snow_classifier.pt")


def load_classifier(path):
    checkpoint = torch.load(path)
    model = make_classifier(checkpoint["hidden"])
    model.load_state_dict(checkpoint["model"])
    model.eval()

    def probability_of_snow(stations):
        scaled_stations = (torch.as_tensor(stations, dtype=torch.float32) - checkpoint["mean"]) / checkpoint["std"]
        with torch.no_grad():
            return torch.sigmoid(model(scaled_stations)).squeeze(1)

    return probability_of_snow


probability_of_snow = load_classifier(SCRATCH / "snow_classifier.pt")
planned = np.array([[69.0, 20.0, 400.0, 30.0]], dtype="float32")
print("planned station:", round(probability_of_snow(planned).item(), 3))

planned station: 0.913


The planned station, at latitude 69 and 400 meters up, gets a probability of snow on 1 May of 0.913.
`load_classifier` needed the file and `make_classifier`, and nothing from the training run.

### The same project in Keras with the scaling inside the model

A `Normalization` layer, adapted on the training stations, learns the same means and standard
deviations as the NumPy cell above, and becomes the model's first layer, so the model takes features
as they come. `EarlyStopping` watches the validation loss, stops after 15 epochs without a better
one, and with `restore_best_weights=True` puts back the best epoch's weights. The accuracy metric is
`BinaryAccuracy(threshold=0.0)`, because the model gives logits, and the Common errors show what the
plain `"accuracy"` does with them.

In [10]:
normalization = keras.layers.Normalization()
normalization.adapt(features[train_rows])
print(np.allclose(normalization.mean.numpy().ravel(), mean),
      np.allclose(np.sqrt(normalization.variance.numpy().ravel()), std))

keras.utils.set_random_seed(0)
net_keras = keras.Sequential([
    keras.Input((4,)),
    normalization,
    keras.layers.Dense(16, activation="relu"),
    keras.layers.Dense(1),
])
net_keras.compile(
    optimizer=keras.optimizers.Adam(0.01),
    loss=keras.losses.BinaryCrossentropy(from_logits=True),
    metrics=[keras.metrics.BinaryAccuracy(name="accuracy", threshold=0.0)],
)
history = net_keras.fit(
    features[train_rows], snow_on_may_1[train_rows],
    validation_data=(features[valid_rows], snow_on_may_1[valid_rows]),
    epochs=200, batch_size=32, verbose=0,
    callbacks=[keras.callbacks.EarlyStopping(monitor="val_loss", patience=15, restore_best_weights=True)],
)
print("epochs run", len(history.history["loss"]), "| best epoch", int(np.argmin(history.history["val_loss"])) + 1)
test_results = net_keras.evaluate(features[test_rows], snow_on_may_1[test_rows], verbose=0, return_dict=True)
print("test accuracy", round(test_results["accuracy"], 3))

True True
epochs run 26 | best epoch 11
test accuracy 0.82


The two arrays match, `True True`. Training stopped after 26 epochs, 15 after the best one, epoch 11,
and the restored weights get 0.82 of the test stations right, the share the PyTorch network got.

In [11]:
net_keras.save(SCRATCH / "snow_classifier.keras")
loaded_keras = keras.models.load_model(SCRATCH / "snow_classifier.keras")
print(np.array_equal(loaded_keras(features[test_rows]).numpy(), net_keras(features[test_rows]).numpy()))
print("planned station:", round(float(keras.ops.sigmoid(loaded_keras(planned))[0, 0]), 3))

True
planned station: 0.926


The loaded model gives the same logits as `net_keras`, and it answers for the planned station from
its features as they come, because its first layer holds the scaling. It says 0.926 where the PyTorch
network said 0.913, since the two started from different random weights, saw the training stations
in different orders and kept the weights of different epochs, 14 and 11.

## Your turn

Four tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than
getting there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/pytorch-and-tensorflow-deep-dive/15-a-complete-project-solutions.ipynb).

**1.** Train a PyTorch classifier with 32 hidden units with `make_classifier` and `fit`, from `torch.manual_seed(0)`, and print its validation accuracy beside `net`'s.

In [12]:
# your code here

**2.** Print the share of test stations that `net` says have snow, beside the share that do.

In [13]:
# your code here

**3.** Use `probability_of_snow` to answer for three planned stations at once: (66.0, 14.0, 50.0, 2.0), (70.5, 25.0, 250.0, 60.0) and (62.0, 9.0, 1100.0, 90.0).

In [14]:
# your code here

**4.** Count the test stations on which `net` and `net_keras` give different answers.

In [15]:
# your code here

## Common errors

### No error, and an accuracy that compares logits with 0.5

Both classifiers in this notebook give logits. A logit of 0 is a probability of one half, and a logit
of 0.5 is a probability of about 0.62, so counting a station as snow only when its logit is above 0.5
asks for a probability of 0.62 where one half was meant. The training stations show it, since the
test stations are measured only once.

In [16]:
with torch.no_grad():
    above_half = ((net(X_train) > 0.5).float() == y_train).float().mean().item()
print("PyTorch, logits above 0  ", round(accuracy(net, X_train, y_train), 3))
print("PyTorch, logits above 0.5", round(above_half, 3))

net_keras.compile(loss=keras.losses.BinaryCrossentropy(from_logits=True), metrics=["accuracy"])
plain = net_keras.evaluate(features[train_rows], snow_on_may_1[train_rows], verbose=0, return_dict=True)["accuracy"]
keras_zero = float(((net_keras(features[train_rows]).numpy() > 0).ravel() == snow_on_may_1[train_rows]).mean())
keras_above_half = float(((net_keras(features[train_rows]).numpy() > 0.5).ravel() == snow_on_may_1[train_rows]).mean())
print('Keras, "accuracy"        ', round(plain, 3))
print("Keras, logits above 0    ", round(keras_zero, 3))
print("Keras, logits above 0.5  ", round(keras_above_half, 3))

PyTorch, logits above 0   0.869
PyTorch, logits above 0.5 0.865
Keras, "accuracy"         0.865
Keras, logits above 0     0.869
Keras, logits above 0.5   0.865


Compared with 0.5, the PyTorch network's accuracy on the training stations falls from 0.869 to
0.865, 3 stations in 800. Keras's plain `"accuracy"` reports 0.865 for a model whose accuracy
compared with 0 is 0.869, and 0.865 is what comparing its logits with 0.5 gives, so it made the same
comparison, with a threshold meant for probabilities. Compare logits with 0, or give
`BinaryAccuracy` `threshold=0.0`, as the worked examples do.

### No error, and a planned station predicted from unscaled features

The planned station's features given straight to `net`, without the training stations' scaling.

In [17]:
with torch.no_grad():
    unscaled = torch.sigmoid(net(torch.from_numpy(planned))).item()
print("unscaled:", round(unscaled, 3), "| scaled:", round(probability_of_snow(planned).item(), 3))

unscaled: 1.0 | scaled: 0.913


The raw features are far larger than the standardized ones the network learned from, 400 meters
where a standardized elevation averages 0, so the logit is large and the probability rounds to 1.0,
where the scaled answer is 0.913. Nothing raises an error. `load_classifier` scales every station it
is given, which is why the scaling is saved in the same file as the network.

### `RuntimeWarning: invalid value encountered in divide`

The planned station standardized with its own mean and standard deviation, as if it were a new
dataset of one.

In [18]:
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always", RuntimeWarning)
    own_scaling = (planned - planned.mean(axis=0)) / planned.std(axis=0)
for warning in caught:
    print(f"{warning.category.__name__}: {warning.message}")
print(own_scaling)
with torch.no_grad():
    print(torch.sigmoid(net(torch.from_numpy(own_scaling))))

[[nan nan nan nan]]
tensor([[nan]])


A single station's mean is the station itself, so every feature minus its mean is 0, the standard
deviation is 0 as well, and 0 divided by 0 is `nan`, which NumPy warns about and the network carries
through to its answer. Scale a new station with the training stations' `mean` and `std`, as
`load_classifier` does.

In [19]:
shutil.rmtree(SCRATCH)
print("scratch removed:", not SCRATCH.exists())

scratch removed: True


## Recap

- You can split the data once, before anything is fitted, and work out the baseline that the finished
  model's test accuracy is read against.
- You can learn the scaling from the training stations, choose when to stop on the validation
  stations, and measure the test stations once.
- You can save a PyTorch classifier with the numbers it scales with, and load it to answer for a new
  station.
- You can build the same project in Keras, with the scaling inside the model and the accuracy
  measured from logits.

## What is next

The **LightGBM and XGBoost, Deep Dive** guide trains gradient-boosted trees on tables of features like
these stations, and the library's front page lists every guide.

---

&#8592; **Previous:** [Saving and Loading Models](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/pytorch-and-tensorflow-deep-dive/14-saving-and-loading-models.ipynb)  &nbsp;·&nbsp;  [PyTorch and TensorFlow, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/pytorch-and-tensorflow-deep-dive.html)
